# 🛢️ Assignment — Unidad 2: El Problema del Derrame de Petróleo

---

> **Curso:** Estructuras de Datos y Algoritmos  
> **Unidad:** 2 — Estructuras de Datos  
> **Entrega:** Subir este archivo `.ipynb` al aula virtual con el nombre `A2_<apellido>_<nombre>.ipynb`

---

## 📖 Contexto del problema

En abril de 2010, el accidente de la plataforma **Deepwater Horizon** en el Golfo de México provocó el derrame de petróleo más grande de la historia. El crudo se expandió durante semanas formando manchas en la superficie del océano.

Para estudiar la conectividad de estos derrames, los oceanógrafos modelan el océano como una **grilla de N×N celdas**. Cada celda puede estar:

- 🟦 **Limpia** — no contiene petróleo
- 🟫 **Contaminada** — contiene petróleo

```
  0   1   2   3   4       ← columna
┌───┬───┬───┬───┬───┐
│ . │ . │ █ │ . │ . │  0  ← fila
├───┼───┼───┼───┼───┤
│ . │ █ │ █ │ . │ . │  1
├───┼───┼───┼───┼───┤
│ . │ █ │ . │ . │ █ │  2
├───┼───┼───┼───┼───┤
│ . │ █ │ █ │ █ │ █ │  3
├───┼───┼───┼───┼───┤
│ . │ . │ █ │ . │ . │  4
└───┴───┴───┴───┴───┘
  █ = contaminada    . = limpia
```

Dos celdas contaminadas forman parte de la **misma mancha** si están conectadas horizontal o verticalmente (no diagonal).

### Preguntas que queremos responder

1. **¿Cuántas manchas independientes** hay en la grilla?
2. **¿Cuál es el tamaño** (en celdas) de la mancha más grande?
3. **¿Están dos celdas** en la misma mancha?
4. **¿La mancha llega a la costa?** (¿toca algún borde de la grilla?)

---

## 🎯 Tu tarea

Se te entrega la clase `WeightedQuickUnion` **ya implementada**. Tu trabajo es construir la clase `OilSpill` que usa esta estructura para responder las preguntas anteriores.

**No debes modificar** `WeightedQuickUnion` ni las celdas de tests.

---

## ⚙️ Celda 1 — Setup (ejecutar primero, no modificar)

In [ ]:
# ════════════════════════════════════════════════════════════
#  CÓDIGO BASE — NO MODIFICAR
# ════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from abc import ABC, abstractmethod


# ── TDA base ────────────────────────────────────────────────
class UnionFind(ABC):
    @abstractmethod
    def union(self, p, q): pass
    @abstractmethod
    def find(self, p): pass
    def connected(self, p, q): return self.find(p) == self.find(q)
    @abstractmethod
    def count(self): pass


# ── Implementación dada: Weighted Quick Union ────────────────
class WeightedQuickUnion(UnionFind):
    """
    Weighted Quick Union.
    Profundidad máxima garantizada: O(log N)
    Costo de union y find: O(log N)
    """
    def __init__(self, n):
        self._id   = list(range(n))
        self._size = [1] * n
        self._count = n

    def _root(self, p):
        while self._id[p] != p:
            p = self._id[p]
        return p

    def find(self, p):
        return self._root(p)

    def union(self, p, q):
        rp, rq = self._root(p), self._root(q)
        if rp == rq: return
        if self._size[rp] < self._size[rq]:
            self._id[rp] = rq
            self._size[rq] += self._size[rp]
        else:
            self._id[rq] = rp
            self._size[rp] += self._size[rq]
        self._count -= 1

    def count(self):
        return self._count

    def component_size(self, p):
        """Retorna el tamaño del componente al que pertenece p."""
        return self._size[self._root(p)]


# ── Función de visualización (ya implementada) ───────────────
def visualizar_grilla(grid, titulo="Grilla", manchas=None):
    """
    Muestra la grilla coloreando cada mancha de un color distinto.
    grid   : lista de listas de bool (True = contaminada)
    manchas: dict {(fila,col): id_mancha} opcional, para colorear por componente
    """
    n = len(grid)
    fig, ax = plt.subplots(figsize=(min(10, n * 0.9 + 1), min(10, n * 0.9 + 1)))

    if manchas:
        ids_unicos = list(set(manchas.values()))
        cmap = plt.cm.tab20(np.linspace(0, 1, max(len(ids_unicos), 2)))
        color_map = {mid: cmap[i] for i, mid in enumerate(ids_unicos)}

    for r in range(n):
        for c in range(n):
            if grid[r][c]:  # contaminada
                if manchas and (r, c) in manchas:
                    color = color_map[manchas[(r, c)]]
                else:
                    color = '#2d3436'
                rect = plt.Rectangle([c, n - 1 - r], 1, 1,
                                     facecolor=color, edgecolor='white', lw=0.5)
            else:  # limpia
                rect = plt.Rectangle([c, n - 1 - r], 1, 1,
                                     facecolor='#dfe6e9', edgecolor='white', lw=0.5)
            ax.add_patch(rect)

    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_xticks(np.arange(n) + 0.5)
    ax.set_xticklabels(range(n))
    ax.set_yticks(np.arange(n) + 0.5)
    ax.set_yticklabels(range(n - 1, -1, -1))
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.set_xlabel("columna")
    ax.set_ylabel("fila")
    plt.tight_layout()
    plt.show()


print("✅ Setup completo — WeightedQuickUnion y utilidades cargadas")

---

## 📐 Celda 2 — Representación de la grilla

Antes de implementar, necesitas entender cómo mapear una grilla 2D a un arreglo 1D (que es lo que usa `WeightedQuickUnion`).

```
Grilla N×N  →  arreglo de tamaño N²

celda (fila, col)  →  índice = fila * N + col

Ejemplo N=5:
  (0,0)→0   (0,1)→1   (0,2)→2   (0,3)→3   (0,4)→4
  (1,0)→5   (1,1)→6   (1,2)→7   (1,3)→8   (1,4)→9
  ...
  (4,0)→20  ...                            (4,4)→24
```

Los **vecinos** de `(r, c)` son:
- Arriba:    `(r-1, c)`
- Abajo:     `(r+1, c)`
- Izquierda: `(r, c-1)`
- Derecha:   `(r, c+1)`

Solo se conectan vecinos que **existan dentro de la grilla** y que **estén contaminados**.

---

## 💻 Celda 3 — Tu implementación

Completa todos los métodos marcados con `# TODO`.

In [ ]:
# ════════════════════════════════════════════════════════════
#  TU IMPLEMENTACIÓN — completa los TODO
# ════════════════════════════════════════════════════════════

class OilSpill:
    """
    Modela un derrame de petróleo en una grilla N×N.
    Usa WeightedQuickUnion para responder consultas de conectividad.

    Parámetro:
        grid : lista de listas de bool
               grid[r][c] == True  →  celda (r,c) está contaminada
               grid[r][c] == False →  celda (r,c) está limpia
    """

    def __init__(self, grid: list):
        self.grid = grid
        self.n    = len(grid)          # tamaño de la grilla (N×N)

        # TODO 1: Crea un WeightedQuickUnion con N*N elementos
        self.uf = # TODO

        # TODO 2: Recorre todas las celdas contaminadas.
        #         Para cada una, únela con sus vecinos contaminados
        #         (arriba, abajo, izquierda, derecha).
        #         Usa self._idx(r, c) para convertir (fila, col) → índice.
        #
        # Pista:
        #   for r in range(self.n):
        #       for c in range(self.n):
        #           if self.grid[r][c]:          ← celda contaminada
        #               for (nr, nc) in vecinos: ← define los 4 vecinos
        #                   if válido y contaminado:
        #                       self.uf.union(...)

        # TODO: escribe el código aquí


    # ── Método auxiliar (ya implementado) ───────────────────
    def _idx(self, r: int, c: int) -> int:
        """Convierte coordenadas (fila, col) al índice del arreglo 1D."""
        return r * self.n + c


    # ── Métodos que debes implementar ───────────────────────

    def num_manchas(self) -> int:
        """
        TODO 3: Retorna el número de manchas (componentes conectados
        de celdas contaminadas).

        Cuidado: self.uf.count() cuenta TODOS los componentes,
        incluyendo las celdas limpias. Debes contar solo
        los componentes que contienen al menos una celda contaminada.

        Pista: usa un set() de raíces de celdas contaminadas.
        """
        # TODO
        pass


    def mancha_mas_grande(self) -> int:
        """
        TODO 4: Retorna el tamaño (número de celdas) de la mancha
        más grande.
        Retorna 0 si no hay celdas contaminadas.

        Pista: itera por las celdas contaminadas y usa
               self.uf.component_size(idx) para saber el tamaño
               del componente de cada una.
        """
        # TODO
        pass


    def misma_mancha(self, r1: int, c1: int, r2: int, c2: int) -> bool:
        """
        TODO 5: Retorna True si las celdas (r1,c1) y (r2,c2) pertenecen
        a la misma mancha.
        Retorna False si alguna de las dos celdas está limpia.
        """
        # TODO
        pass


    def llega_al_borde(self, r: int, c: int) -> bool:
        """
        TODO 6: Retorna True si la mancha que contiene la celda (r,c)
        toca algún borde de la grilla (fila 0, fila n-1, col 0 o col n-1).
        Retorna False si la celda (r,c) está limpia.

        Pista: itera por TODAS las celdas del borde y verifica si
               alguna contaminada está en el mismo componente que (r,c).
        """
        # TODO
        pass


    def mapa_manchas(self) -> dict:
        """
        TODO 7: Retorna un diccionario  { (r, c): id_mancha }
        con todas las celdas contaminadas, donde id_mancha es
        el identificador del componente (la raíz de su árbol).
        Se usa para visualizar la grilla con colores por mancha.
        """
        # TODO
        pass


print("Clase OilSpill definida. Continúa con las celdas de demostración y tests.")

---

## 🔍 Celda 4 — Exploración y demostración

Usa esta celda para probar tu implementación antes de los tests formales.

In [ ]:
# Grilla de ejemplo del enunciado (5×5)
# True = contaminada, False = limpia
grid_ejemplo = [
    [False, False, True,  False, False],
    [False, True,  True,  False, False],
    [False, True,  False, False, True ],
    [False, True,  True,  True,  True ],
    [False, False, True,  False, False],
]

oil = OilSpill(grid_ejemplo)

print("=" * 40)
print("  Análisis del derrame")
print("=" * 40)
print(f"Número de manchas     : {oil.num_manchas()}")
print(f"Mancha más grande     : {oil.mancha_mas_grande()} celdas")
print(f"¿(0,2) y (4,2) misma? : {oil.misma_mancha(0, 2, 4, 2)}")
print(f"¿(2,4) y (3,3) misma? : {oil.misma_mancha(2, 4, 3, 3)}")
print(f"¿Mancha (0,2) al borde: {oil.llega_al_borde(0, 2)}")
print(f"¿Mancha (2,4) al borde: {oil.llega_al_borde(2, 4)}")

# Visualización
manchas = oil.mapa_manchas()
visualizar_grilla(grid_ejemplo, titulo="Derrame — manchas coloreadas", manchas=manchas)

In [ ]:
# Experimenta aquí con tu propia grilla
# Modifica los True/False para crear distintos patrones

mi_grid = [
    [True,  False, False, True ],
    [True,  False, False, True ],
    [False, False, True,  True ],
    [False, True,  True,  False],
]

mi_oil = OilSpill(mi_grid)
print(f"Manchas: {mi_oil.num_manchas()}  |  Más grande: {mi_oil.mancha_mas_grande()} celdas")
visualizar_grilla(mi_grid, titulo="Mi grilla personalizada", manchas=mi_oil.mapa_manchas())

---

## 🧪 Celda 5 — Tests automáticos *(no modificar)*

In [ ]:
# ════════════════════════════════════════════════════════════
#  TESTS AUTOMÁTICOS — NO MODIFICAR
# ════════════════════════════════════════════════════════════

def ejecutar_tests():
    resultados = []   # (nombre, passed, mensaje)

    def caso(nombre, condicion, detalle=""):
        resultados.append((nombre, condicion, detalle))

    # ── Grilla de referencia ──────────────────────────────────
    #
    #  . . █ . .      mancha A: (0,2)(1,1)(1,2)(2,1)(3,1)(3,2)(3,3)(3,4)(4,2) → 9 celdas
    #  . █ █ . .      mancha B: (2,4)                                          → 1 celda
    #  . █ . . █      total manchas = 2
    #  . █ █ █ █
    #  . . █ . .
    g = [
        [False, False, True,  False, False],
        [False, True,  True,  False, False],
        [False, True,  False, False, True ],
        [False, True,  True,  True,  True ],
        [False, False, True,  False, False],
    ]
    o = OilSpill(g)

    # T1 — num_manchas
    caso("T1 · num_manchas básico",
         o.num_manchas() == 2,
         f"esperado 2, obtenido {o.num_manchas()}")

    # T2 — mancha_mas_grande
    caso("T2 · mancha_mas_grande",
         o.mancha_mas_grande() == 9,
         f"esperado 9, obtenido {o.mancha_mas_grande()}")

    # T3 — misma_mancha: True
    caso("T3 · misma_mancha → True",
         o.misma_mancha(0, 2, 4, 2) == True,
         "(0,2) y (4,2) deben estar en la misma mancha")

    # T4 — misma_mancha: False (manchas distintas)
    caso("T4 · misma_mancha → False (manchas distintas)",
         o.misma_mancha(0, 2, 2, 4) == False,
         "(0,2) y (2,4) son manchas distintas")

    # T5 — misma_mancha: celda limpia
    caso("T5 · misma_mancha → False (celda limpia)",
         o.misma_mancha(0, 0, 0, 2) == False,
         "(0,0) está limpia, debe retornar False")

    # T6 — llega_al_borde: mancha grande sí toca borde
    caso("T6 · llega_al_borde → True",
         o.llega_al_borde(0, 2) == True,
         "la mancha grande toca fila 0")

    # T7 — llega_al_borde: mancha pequeña no toca borde
    caso("T7 · llega_al_borde → False",
         o.llega_al_borde(2, 4) == False,
         "la mancha (2,4) no toca ningún borde")

    # T8 — mapa_manchas: todas las celdas contaminadas están en el mapa
    mapa = o.mapa_manchas()
    contaminadas = [(r, c) for r in range(5) for c in range(5) if g[r][c]]
    caso("T8 · mapa_manchas completo",
         all(celda in mapa for celda in contaminadas),
         "todas las celdas contaminadas deben estar en el mapa")

    # T9 — mapa_manchas: celdas limpias NO están en el mapa
    limpias = [(r, c) for r in range(5) for c in range(5) if not g[r][c]]
    caso("T9 · mapa_manchas no incluye celdas limpias",
         all(celda not in mapa for celda in limpias),
         "celdas limpias no deben aparecer en el mapa")

    # T10 — grilla vacía
    g_vacia = [[False]*3 for _ in range(3)]
    o_v = OilSpill(g_vacia)
    caso("T10 · grilla sin contaminación",
         o_v.num_manchas() == 0 and o_v.mancha_mas_grande() == 0,
         f"manchas={o_v.num_manchas()}, grande={o_v.mancha_mas_grande()}")

    # T11 — grilla completamente contaminada 3×3
    g_full = [[True]*3 for _ in range(3)]
    o_f = OilSpill(g_full)
    caso("T11 · grilla 3×3 totalmente contaminada",
         o_f.num_manchas() == 1 and o_f.mancha_mas_grande() == 9,
         f"manchas={o_f.num_manchas()}, grande={o_f.mancha_mas_grande()}")

    # T12 — celdas aisladas
    g_aislado = [
        [True,  False, True ],
        [False, False, False],
        [True,  False, True ],
    ]
    o_a = OilSpill(g_aislado)
    caso("T12 · 4 celdas aisladas → 4 manchas",
         o_a.num_manchas() == 4,
         f"esperado 4, obtenido {o_a.num_manchas()}")

    # T13 — misma_mancha simétrico
    caso("T13 · misma_mancha es simétrico",
         o.misma_mancha(0, 2, 4, 2) == o.misma_mancha(4, 2, 0, 2),
         "misma_mancha(p,q) debe ser igual a misma_mancha(q,p)")

    # ── Imprimir resultados ───────────────────────────────────
    print("=" * 52)
    print("  RESULTADOS DE TESTS")
    print("=" * 52)
    aprobados = 0
    for nombre, ok, detalle in resultados:
        estado = "✅ PASS" if ok else "❌ FAIL"
        msg = f"  {estado}  {nombre}"
        if not ok:
            msg += f"\n         → {detalle}"
        print(msg)
        if ok: aprobados += 1

    total = len(resultados)
    print("=" * 52)
    print(f"  Resultado: {aprobados}/{total} tests pasados")
    print("=" * 52)
    return aprobados, total, resultados


try:
    aprobados, total, _ = ejecutar_tests()
except Exception as e:
    print(f"⚠️  Error al ejecutar tests: {e}")
    print("    Verifica que todos los TODO estén implementados.")

---

## ✍️ Celda 6 — Preguntas de reflexión

Responde en las celdas de texto (doble clic para editar).

**Pregunta 1:** En el método `__init__`, recorres la grilla y llamas a `union()` para cada par de vecinos contaminados. ¿Cuántas veces como máximo puede llamarse `union()` en una grilla N×N? Expresa tu respuesta en función de N.

> *Tu respuesta aquí...*

**Pregunta 2:** El método `num_manchas()` no puede usar directamente `self.uf.count()`. ¿Por qué? ¿Qué contaría de más si lo usaras directamente?

> *Tu respuesta aquí...*

**Pregunta 3:** ¿Qué complejidad tiene tu método `llega_al_borde()`? Exprésala en función de N y justifica.

> *Tu respuesta aquí...*

---

## ✅ Celda 7 — Checklist de entrega

Antes de subir verifica:

- [ ] Ejecuté `Kernel → Restart & Run All` sin errores
- [ ] Mis tests muestran el puntaje final
- [ ] Las 3 preguntas de reflexión están respondidas
- [ ] El archivo se llama `A2_<apellido>_<nombre>.ipynb`

---

**Nombre completo:** ___________________________  
**Fecha de entrega:** ___________________________